# K-Means Customer Segmentation (Retail Bank) – Solution

**Short name (GitHub):** `KMBank`

Worked answers for `KMBank_Practice_Skeleton.ipynb`. Numbers below use `KMBank.KMeans` (or sklearn if present) on **scaled** features, `n_init=10`, `random_state=0`.

| Card | k | Inertia (scaled) | Purity | ARI |
|------|---|------------------|--------|-----|
| Toy DTI × util | 3 | — | ~0.60–0.70 | coarse buckets |
| Full 10-feature book | 5 | 4,670 | 0.978 | 0.945 |
| k=3 on full book | 3 | 7,246 | 0.600 | — |
| k=8 on full book | 8 | 4,113 | 0.982 | extra splits |

Cluster ids are **not** persona names. Map each centroid with a majority vote (or a business glossary) before you brief Retail or Credit Risk.


## Inline cheat-sheet (keep this cell visible)

See also **`KMBank_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Sample vs feature | row = one customer; column = score, DTI, util, deposits, … |
| Scale first | \(x'=(x-\mu)/\sigma\) — score 575–785 must not dominate util 0–1 |
| Distance | \(d(x,c)=\sqrt{\sum_j(x'_j-c_j)^2}\) on the **scaled** row |
| Assign / update | nearest centroid; centroid = mean of its rows |
| Inertia | \(J=\sum_i\|x_i-c_{\ell_i}\|^2\) on the scaled matrix |
| Elbow | plot \(J(k)\); domain k=5 (five retail personas) |
| Purity / ARI | majority-map cluster → planted persona, then score |
| Inference | scale a new applicant with the **training** \(\mu,\sigma\), then `.predict` |

**Order:** scale → `fit` → map names → `predict` new rows with the same scaler.


## Desired outcome

![flowchart](kmbank_flowchart.png)

1. Load the book (samples × features). Drop `customer_id` and the persona label from `X`.
2. Scale columns. Credit score and utilization do not share a unit.
3. Choose \(k\) from the retail glossary (five personas) and confirm with an elbow.
4. Place centroids, assign, update, until the shift is below `tol`.
5. Profile each centroid in **original units** (the chart a branch manager can read).
6. Map cluster ids → persona names. Score only if a label exists.
7. Score four new applicants. Simulate k / n / noise / n_init.


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.cluster import KMeans, MiniBatchKMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.metrics import adjusted_rand_score
    HAS_SK = True
except ImportError:
    HAS_SK = False
    print("sklearn missing — KMBank.KMeans + numpy scaler / PCA")

from KMBank import (
    KMeans as KMScratch,
    assign,
    inertia,
    map_clusters_to_truth,
    adjusted_rand,
    elbow_curve,
)

FEATS = [
    "credit_score", "income_k", "dti", "card_util", "deposit_k",
    "loan_k", "n_products", "tenure_mo", "late30_12m", "logins_30d",
]
PERSONAS = [
    "Prime Transactor",
    "Mass-Market Saver",
    "Card Revolver",
    "Stressed Book",
    "Affluent Relationship",
]
plt.rcParams["figure.figsize"] = (7, 4)
print("sklearn available:", HAS_SK)


## 1. Unsupervised clustering in a bank

The book is not labeled for this exercise. K-means answers two questions: how many groups (`k`), and what “similar” means (Euclidean distance to a centroid after z-scoring). Training = assign + update. Inference = nearest centroid. A planted persona column exists only so we can *grade* the clustering.

## 2. Load the retail book and scale it

2,000 customers, 400 in each of five personas. Features mix 300–850 scores with 0–1 utilization — scale before you cluster.

In [ ]:
book = pd.read_csv("data/bank_customers.csv")
print(book.shape)
print(book["segment_name"].value_counts())
print(book[FEATS].describe().round(2).T)

X_raw = book[FEATS].to_numpy(dtype=float)
y = book["segment"].to_numpy(int)          # planted persona — NOT passed to fit
ids = book["customer_id"].to_numpy()
print("X_raw", X_raw.shape, "labels held out of fit")


In [ ]:
def scale_fit(X):
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=0)
    sd = np.where(sd == 0, 1.0, sd)
    return (X - mu) / sd, mu, sd

def scale_apply(X, mu, sd):
    return (X - mu) / sd

X, mu, sd = scale_fit(X_raw)
print("scaled means ~0", np.round(X.mean(0)[:4], 3))
print("scaled stds  ~1", np.round(X.std(0)[:4], 3))
print("raw score span", X_raw[:, 0].min(), "→", X_raw[:, 0].max(),
      "vs util span", X_raw[:, 3].min(), "→", X_raw[:, 3].max())


## 3. Look at the book before you cluster

In [ ]:
fig, ax = plt.subplots()
sc = ax.scatter(book["card_util"], book["credit_score"], c=y,
                cmap="tab10", s=12, alpha=0.7)
ax.set_xlabel("Card utilization")
ax.set_ylabel("Credit score")
ax.set_title("Retail book — planted personas (color = segment, not used by k-means)")
plt.colorbar(sc, ax=ax, ticks=range(5), label="persona id")
plt.tight_layout()
plt.savefig("kmbank_samples.png", dpi=120, bbox_inches="tight")
plt.show()
print(book.groupby("segment_name")[["credit_score", "card_util", "late30_12m", "deposit_k"]].median().round(2))


## 4. From-scratch k-means on a 2-feature toy (DTI × util)

Banking analog of the Iris warm-up. Four functions: place, assign, update, loop.

In [ ]:
# Warm-up: two features that a collections analyst already plots — DTI and utilization.
toy = X[:, [2, 3]]   # already scaled
C0 = toy[[10, 200, 800]]   # three rows as a crude start; loop will move them

def random_centroids(X, k, seed=0):
    r = np.random.default_rng(seed)
    return X[r.choice(len(X), size=k, replace=False)].copy()

def assign_labels(X, centroids):
    d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return d2.argmin(axis=1)

def update_centroids(X, labels, k):
    C = np.zeros((k, X.shape[1]))
    for j in range(k):
        mask = labels == j
        C[j] = X[mask].mean(axis=0) if mask.any() else X[np.random.randint(len(X))]
    return C

def kmeans_loop(X, k, max_iter=40, seed=0, tol=1e-4):
    C = random_centroids(X, k, seed)
    history = []
    for t in range(1, max_iter + 1):
        labels = assign_labels(X, C)
        new_C = update_centroids(X, labels, k)
        shift = float(np.linalg.norm(new_C - C))
        history.append((t, float(((X - new_C[labels]) ** 2).sum()), shift))
        if shift < tol:
            C = new_C
            break
        C = new_C
    return C, assign_labels(X, C), history

Ct, labt, hist = kmeans_loop(toy, k=3, seed=0)
print("toy k=3 inertia", round(hist[-1][1], 2), "iters", hist[-1][0])
# Coarse 3-bucket truth from utilization+DTI: low / mid / high risk-ish
y_toy = np.where(book["card_util"] + book["dti"] < 0.45, 0,
                 np.where(book["card_util"] + book["dti"] < 0.95, 1, 2))
_, _, pur_toy = map_clusters_to_truth(labt, y_toy, 3)
print("toy purity vs coarse risk buckets", round(pur_toy, 3))


## 5. Full 10-feature book — k=5

In [ ]:
# Prefer sklearn when present; otherwise the bundled class (same API).
KM = KMeans if HAS_SK else KMScratch
model = KM(n_clusters=5, n_init=10, random_state=0)
model.fit(X)
print("inertia", round(model.inertia_, 2), "sizes", np.bincount(model.labels_))
print("n_iter", getattr(model, "n_iter_", "n/a"))


## 6. Centroids as prototype customers

A centroid lives in the same 10-D space as a row, so it *is* a customer profile. Back-transform to dollars and percents.

In [ ]:
# Back-transform centroids into the units a relationship manager reads.
C_raw = model.cluster_centers_ * sd + mu
cent = pd.DataFrame(C_raw, columns=FEATS)
cent.index = [f"cluster_{j}" for j in range(5)]
print(cent.round(2).T)

show = ["credit_score", "income_k", "card_util", "deposit_k", "late30_12m", "n_products"]
Cn = cent[show].copy()
Cn = (Cn - Cn.min()) / (Cn.max() - Cn.min())
ax = Cn.T.plot(kind="bar", figsize=(9, 4))
ax.set_title("Centroid profiles (min–max across the 5 groups)")
ax.set_ylabel("relative level")
plt.tight_layout()
plt.savefig("kmbank_centroids.png", dpi=120, bbox_inches="tight")
plt.show()


## 7. Map cluster ids to persona names + score

In [ ]:
mapped, mapping, purity = map_clusters_to_truth(model.labels_, y, k=5)
ari = adjusted_rand(y, model.labels_) if not HAS_SK else (
    __import__("sklearn.metrics", fromlist=["adjusted_rand_score"]).adjusted_rand_score(y, model.labels_)
    if HAS_SK else adjusted_rand(y, model.labels_)
)
ari = adjusted_rand(y, model.labels_)
print("cluster → planted persona id", mapping)
print("cluster → name", {j: PERSONAS[mapping[j]] for j in mapping})
print("purity", round(purity, 4), "ARI", round(ari, 4))

ct = pd.crosstab(pd.Series([PERSONAS[i] for i in y], name="persona"),
                 pd.Series(model.labels_, name="cluster"))
print(ct)

fig, ax = plt.subplots(figsize=(6.2, 4.2))
im = ax.imshow(ct.values, cmap="Blues")
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(ct.columns)
ax.set_yticklabels(ct.index, fontsize=8)
ax.set_xlabel("cluster id"); ax.set_title("Persona vs raw cluster id")
for i in range(5):
    for j in range(5):
        ax.text(j, i, int(ct.values[i, j]), ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("kmbank_crosstab.png", dpi=120, bbox_inches="tight")
plt.show()


## 8. Elbow method

Inertia always drops with k. The retail glossary already says five personas; the elbow should be *compatible*, not a substitute for that story.

In [ ]:
ks = list(range(1, 11))
inertias = []
for kk in ks:
    m = (KMeans if HAS_SK else KMScratch)(n_clusters=kk, n_init=4, random_state=0)
    m.fit(X)
    inertias.append(m.inertia_)

fig, ax = plt.subplots()
ax.plot(ks, inertias, marker="o")
ax.axvline(5, color="crimson", ls="--", label="domain k=5")
ax.set_xlabel("k"); ax.set_ylabel("inertia (scaled)")
ax.set_title("Elbow — retail book")
ax.legend()
plt.tight_layout()
plt.savefig("kmbank_elbow.png", dpi=120, bbox_inches="tight")
plt.show()
print(list(zip(ks, [round(v, 1) for v in inertias])))
print("Drop is steep through k=5, then flattens. Five is the glossary, confirmed by the bend.")


## 9. PCA view (10-D → 2-D)

In [ ]:
def pca2(X):
    Xc = X - X.mean(0)
    _, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    Z = Xc @ Vt[:2].T
    ev = (S[:2] ** 2) / (S ** 2).sum()
    return Z, ev

Z, ev = pca2(X)
print("explained", ev.round(3), "sum", ev.sum().round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axes[0].scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=8, alpha=0.7)
axes[0].set_title("PCA — planted persona")
axes[1].scatter(Z[:, 0], Z[:, 1], c=model.labels_, cmap="tab10", s=8, alpha=0.7)
axes[1].set_title("PCA — k-means cluster")
for ax in axes:
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("kmbank_pca.png", dpi=120, bbox_inches="tight")
plt.show()


## 10. Alternate code that reaches the same idea

In [ ]:
# A. random init vs k-means++
m_rand = KMScratch(n_clusters=5, init="random", n_init=10, random_state=0).fit(X)
m_pp = KMScratch(n_clusters=5, init="k-means++", n_init=10, random_state=0).fit(X)
print("random J", round(m_rand.inertia_, 1), "++ J", round(m_pp.inertia_, 1))

# B. unscaled features — score in the 700s owns Euclidean distance
m_raw = KMScratch(n_clusters=5, n_init=10, random_state=0).fit(X_raw)
print("UNSCALED purity", round(map_clusters_to_truth(m_raw.labels_, y, 5)[2], 3),
      "(expect a drop — score dominates util and DTI)")

# C. fit_predict one-liner
lab_fp = KMScratch(n_clusters=5, n_init=10, random_state=0).fit_predict(X)
print("fit_predict size", np.bincount(lab_fp))

# D. sklearn MiniBatch if present
if HAS_SK:
    mb = MiniBatchKMeans(n_clusters=5, n_init=10, random_state=0, batch_size=256).fit(X)
    print("minibatch purity", round(map_clusters_to_truth(mb.labels_, y, 5)[2], 3))


## 11. More practice

In [ ]:
# P1. Only risk columns: score, dti, util, late30
risk_idx = [0, 2, 3, 8]
m_risk = KMScratch(n_clusters=5, n_init=10, random_state=0).fit(X[:, risk_idx])
print("P1 risk-only purity", round(map_clusters_to_truth(m_risk.labels_, y, 5)[2], 3))

# P2. Wrong k
for kk in (3, 8):
    mk = KMScratch(n_clusters=kk, n_init=5, random_state=0).fit(X)
    print(f"P2 k={kk} J={mk.inertia_:.0f} purity={map_clusters_to_truth(mk.labels_, y, kk)[2]:.3f}")

# P3. Two columns only — util and deposits
m2 = KMScratch(n_clusters=5, n_init=10, random_state=0).fit(X[:, [3, 4]])
print("P3 util+deposits purity", round(map_clusters_to_truth(m2.labels_, y, 5)[2], 3),
      "(Affluent vs Saver still separate; Prime vs Revolver blur)")


## 12. Four new applicants

In [ ]:
# Four applicants who walked in this morning. Same 10 columns, raw units.
# Order: credit_score, income_k, dti, card_util, deposit_k,
#        loan_k, n_products, tenure_mo, late30_12m, logins_30d
new_raw = np.array([
    [790, 102, 0.20, 0.10, 30,  8, 3,  80, 0, 16],   # looks Prime
    [580,  28, 0.55, 0.90,  1, 22, 2,  20, 5,  4],   # looks Stressed
    [760, 170, 0.30, 0.16, 90, 250, 6, 110, 0, 12],  # looks Affluent
    [640,  44, 0.38, 0.55,  8, 20, 3,  40, 2, 18],   # borderline Revolver / Saver
], dtype=float)
new_X = scale_apply(new_raw, mu, sd)
new_ids = model.predict(new_X)
new_names = [PERSONAS[mapping[int(c)]] for c in new_ids]
for i, (cid, name) in enumerate(zip(new_ids, new_names)):
    print(f"applicant {i+1}: cluster {cid} → {name}")

print("Disclaimer: a bin is not an approve/decline. Underwriting still reads the file.")


## 13. Simulation — turn the knobs

In [ ]:
def run_once(k=5, n=2000, noise=0.0, n_init=10, seed=0):
    r = np.random.default_rng(seed)
    idx = r.choice(len(X), size=min(n, len(X)), replace=False)
    Xs = X[idx] + r.normal(0.0, noise, size=(len(idx), X.shape[1]))
    ys = y[idx]
    m = KMScratch(n_clusters=k, n_init=n_init, random_state=seed).fit(Xs)
    _, _, pur = map_clusters_to_truth(m.labels_, ys, k)
    return {"k": k, "n": len(idx), "noise": noise, "n_init": n_init,
            "inertia": m.inertia_, "purity": pur, "ARI": adjusted_rand(ys, m.labels_)}

grid = [
    dict(k=5, n=2000, noise=0.0, n_init=10, seed=0),
    dict(k=3, n=2000, noise=0.0, n_init=10, seed=0),
    dict(k=5, n=400,  noise=0.0, n_init=10, seed=0),
    dict(k=5, n=2000, noise=0.8, n_init=10, seed=0),
    dict(k=5, n=2000, noise=0.0, n_init=1,  seed=0),
]
sim = pd.DataFrame([run_once(**g) for g in grid])
print(sim.round(3))

sweep = pd.DataFrame([run_once(k=kk, n_init=5, seed=0) for kk in range(2, 11)])
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].plot(sweep["k"], sweep["inertia"], marker="o")
axes[0].set_title("Simulation — inertia vs k")
axes[0].set_xlabel("k")
axes[1].plot(sweep["k"], sweep["purity"], marker="o", color="teal")
axes[1].axhline(0.20, color="gray", ls=":", label="chance 1/5")
axes[1].set_title("Simulation — purity vs k")
axes[1].set_xlabel("k"); axes[1].legend()
plt.tight_layout()
plt.savefig("kmbank_simulation.png", dpi=120, bbox_inches="tight")
plt.show()


## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence they should hear |
|----------|----------------|-------------------------------|
| Expert (Credit Risk / quant) | scaled Euclidean, inertia 4,670, ARI 0.945, unscaled ablation | k=5 on z-scored 10 columns recovers the planted book at purity 0.978; skip the scaler and score in the 700s swallows DTI. |
| Technician (CRM / campaign ops) | centroid table in dollars and percents | Five prototype customers. Drop a new row on the nearest prototype; treat the bin, not the cluster id. |
| Executive (Retail / CRO) | what changes in the branch | A first-pass grouping so offers, collections intensity, and relationship coverage are not one-size-fits-all. Not a scorecard. |
| Nonspecialist | no jargon | The bank piles customers who look alike on deposits, card use, and late payments, then names each pile. |

Data literacy: Risk gets the elbow; the branch gets the six-bar prototype chart.

Subject knowledge: do not define DTI or utilization for a credit officer. Do define that cluster `2` is a bin number until the glossary names it “Stressed Book.”

**What not to say**
- “The model declined the loan.” There is no decision threshold.
- “97% accurate like a PD model.” Labels were planted so we could *check* a clustering.
- “k=5 is proven by inertia.” Inertia keeps falling; five is the retail story plus a compatible bend.


## What this model can and cannot do

**Can**
- Group a numeric retail book into compact spherical personas.
- Hand a relationship manager a prototype row in original units.
- Assign a *new* scaled 10-vector to the nearest prototype in milliseconds.
- Warm up campaign design, limit-increase waves, or collections intensity tiers.

**Cannot**
- Replace a bureau scorecard or an IFRS-9 PD.
- Respect compliance rules (fair-lending, adverse action) — a cluster is not a reason code.
- Handle mixed categorical products without encoding.
- Stay stable if you skip the scaler or change column order at inference.

**Top banking applications of the same idea**
1. Retail persona / next-best-offer seeds
2. Card-util vs transactor bins
3. Deposit-heavy vs credit-heavy relationship groups
4. Collections early-stage vs late-stage intensity
5. Branch / region clustering on mix and NPL
6. Small-business cash-flow pattern groups
7. ATM / digital-channel usage clusters
8. Wealth-book share-of-wallet prototypes
9. Alert-queue grouping before a case is opened
10. Color-quantization analog: tier a limit grid

**Anti-applications**
- Approve / decline
- Pricing a revolving APR
- Filing a Suspicious Activity Report
- Anything that needs a probability and a hold-out KS


## Next steps

- Add a categorical product flag (one-hot) and re-scale.
- Try \(k=4\) after merging Prime and Affluent for a cheaper campaign grid.
- GMM if you need soft membership (“60% Revolver / 40% Stressed”).
- Reusable template: `KMBank_Reusable_Template.ipynb`.
- Sister lab on glyphs: `KMDigits`.
